# Fine-tune Qwen2.5-Coder-1.5B para SVG con TRL + QLoRA

**BitNeuronal — Issue Semanal #2**

Este notebook entrena un adaptador LoRA sobre Qwen2.5-Coder-1.5B-Instruct para que genere código SVG.

**Tiempo estimado:** ~40 minutos en GPU T4 (gratis).

**Requisitos:**
1. Cuenta de Hugging Face con un token con permiso **Write** — https://huggingface.co/settings/tokens
2. Runtime de Colab configurado como GPU T4: menú **Runtime → Change runtime type → Hardware accelerator: T4 GPU**

> Para más contexto, lee el tutorial completo en el post y el repo: https://github.com/BitNeuronal/qwen-svg-finetune

## Paso A — Verificar GPU

Antes de instalar nada, confirma que Colab te asignó una T4. Si ves `Tesla T4` con 16 GB, vamos bien.

Si ves `No GPU` o un mensaje de error, cambia el runtime: **Runtime → Change runtime type → T4 GPU** y vuelve a ejecutar.

In [ ]:
!nvidia-smi

## Paso B — Instalar dependencias

Instalamos las librerías en versiones estables. `bitsandbytes` es clave: habilita la cuantización 4-bit (QLoRA) que hace que Qwen 1.5B entre en la T4 con batch decente.

Tarda unos 2–3 minutos la primera vez.

In [ ]:
!pip install -qq \
    "transformers>=4.45.0" \
    "datasets>=3.0.0" \
    "accelerate>=0.33.0" \
    "peft>=0.12.0" \
    "trl>=0.12.0" \
    "bitsandbytes>=0.43.0" \
    "sentencepiece>=0.2.0" \
    "huggingface_hub>=0.25.0" \
    "cairosvg>=2.7.1" 

## Paso C — Login en Hugging Face Hub

Necesitamos autenticarnos en el Hub para descargar el modelo (es público, pero autenticado evita rate-limits) y, al final del tutorial, subir el adaptador entrenado.

### Método recomendado: Colab Secrets

La forma segura de manejar el token es usando el almacén de secretos de Colab (ícono de la **llave** en la barra lateral izquierda), no pegándolo en un widget cada vez.

**Setup (una sola vez):**
1. Abre el panel de Secrets (ícono de llave a la izquierda).
2. **Add new secret** → Name: `HF_TOKEN` → Value: tu token con permiso **Write**.
3. Activa el toggle *"Notebook access"* para este notebook.

Después la celda siguiente lo lee automáticamente. Ventaja: el token nunca aparece en outputs, clipboard ni pantalla. Además queda disponible para los notebooks de evaluación y push al Hub.

### Fallback: widget interactivo

Si el notebook no corre en Colab (ej. Jupyter local, Kaggle), la celda detecta eso y usa `notebook_login()` como fallback — pega el token cuando aparezca el widget.

In [ ]:
from huggingface_hub import login

try:
    # Ruta principal: Colab Secrets
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    login(token=token, add_to_git_credential=True)
    print("✓ Login con HF_TOKEN desde Colab Secrets")
except ImportError:
    # Fuera de Colab: widget interactivo
    from huggingface_hub import notebook_login
    notebook_login()
except Exception as e:
    # Secret no existe o no tiene acceso: fallback al widget
    print(f"⚠ No se pudo leer HF_TOKEN ({e}). Usando login interactivo.")
    from huggingface_hub import notebook_login
    notebook_login()

## Paso D — Cargar el modelo con QLoRA 4-bit

Aquí está la magia: cargamos Qwen2.5-Coder-1.5B **cuantizado a 4 bits** con `bitsandbytes`. Esto reduce la huella del modelo de ~3 GB (fp16) a ~0.8 GB, dejando margen en la T4 para activaciones y optimizer states.

**Configuración QLoRA:**
- `nf4`: Normal Float 4-bit, el cuantizador recomendado en el [paper de QLoRA](https://arxiv.org/abs/2305.14314).
- `double_quant=True`: cuantiza también las constantes de cuantización. Ahorro extra de ~0.4 GB.
- `compute_dtype=float16`: las operaciones se hacen en fp16 (T4 no tiene bf16 a nivel hardware).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Cargando tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"  pad={tokenizer.pad_token}  eos={tokenizer.eos_token}")

print(f"\nCargando modelo en 4-bit (primera vez baja ~1 GB)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # requerido para gradient checkpointing
print(f"\nModelo cargado en: {model.device}  ({model.dtype})")

## Paso E — Cargar el dataset

El dataset `thesantatitan/deepseek-svg-dataset` tiene 4,750 ejemplos de train + 250 de test. La columna `messages` ya viene pre-formateada como conversación chat, lo que TRL consume directamente.

Eliminamos las columnas `prompt` y `completion` porque son redundantes con `messages` (y TRL se quejaría si están).

In [ ]:
from datasets import load_dataset

DATASET_ID = "thesantatitan/deepseek-svg-dataset"
ds = load_dataset(DATASET_ID)

train_ds = ds["train"].remove_columns(["prompt", "completion"])
eval_ds = ds["test"].remove_columns(["prompt", "completion"])

print(f"train: {len(train_ds)} ejemplos")
print(f"eval:  {len(eval_ds)} ejemplos")
print(f"columnas: {train_ds.column_names}")
print(f"\n--- Primer ejemplo (roles) ---")
for m in train_ds[0]["messages"]:
    preview = m["content"][:100].replace("\n", " ")
    print(f"  [{m['role']:9}] {preview}...")

## Paso F — Configurar LoRA

El adaptador LoRA que vamos a entrenar toca los 7 módulos lineales de Qwen: los 4 de atención (`q/k/v/o_proj`) y los 3 del MLP (`gate/up/down_proj`).

**Parámetros:**
- `r=16`: rango del adaptador. Entre 8 y 32 es el rango sano para modelos 1–7B.
- `lora_alpha=32`: factor de escala. La receta típica es `alpha = 2 × r`.
- `dropout=0.05`: regularización ligera.

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
    bias="none",
)

## Paso G — Configurar el entrenamiento (SFTConfig)

Los hiperparámetros más importantes:

- **`per_device_train_batch_size=2` + `gradient_accumulation_steps=4`** → batch efectivo = 8. En T4 con QLoRA 4-bit el batch 2 cabe con `max_length=2048`.
- **`learning_rate=2e-4`** → alto, como corresponde a LoRA.
- **`warmup_steps=20`** → calienta el lr durante ~3% del entrenamiento.
- **`assistant_only_loss=True`** → clave: la loss se calcula solo sobre los tokens del turno assistant (la respuesta del modelo), no sobre system/user.
- **`gradient_checkpointing=True`** → sacrifica ~30% de velocidad por ~40% menos VRAM. Imprescindible en T4 de 16 GB.
- **`fp16=True`** → T4 es Turing (compute capability 7.5), no tiene bf16 a nivel hardware. fp16 + bitsandbytes loss scaling funciona bien acá.

In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./qwen-svg-lora",
    num_train_epochs=1.0,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    max_length=2048,
    packing=False,
    assistant_only_loss=True,            # loss solo en tokens de assistant
    gradient_checkpointing=True,         # OK en CUDA
    bf16=False,                          # T4 no tiene bf16
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="no",
    report_to="none",
    seed=42,
)

## Paso H — Construir el SFTTrainer

`SFTTrainer` envuelve todo: aplica el `chat_template` del tokenizer sobre `messages`, crea el DataLoader, inserta los adaptadores LoRA vía PEFT, y corre el loop de entrenamiento con la config.

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.model.print_trainable_parameters()

## Paso I — Entrenar

Ahora sí, el training real. En T4 con batch=2 y grad_accum=4 sobre 4,750 ejemplos, son ~594 steps totales.

**Tiempo esperado:** 35–45 minutos.

**Qué mirar mientras corre:**
- `loss` debe bajar de forma monótona (con ruido). Arranca cerca de 0.9–1.1 y termina cerca de 0.3–0.5.
- `grad_norm` debe mantenerse estable (típicamente entre 0.1 y 1.0). Si explota a 10+ o aparece NaN, hay problema.
- `learning_rate` sube durante los primeros 20 steps (warmup) y después baja con el cosine scheduler.

> **Tip:** si Colab te desconecta durante el training, el adaptador se pierde. Si vas a dejar el notebook corriendo sin supervisión, pon `save_strategy="steps"` con `save_steps=100` arriba. Para este tutorial mantenemos `"epoch"` porque es un único epoch.

In [ ]:
trainer.train()

## Paso J — Guardar el adaptador localmente

`save_model` guarda **solo** el adaptador LoRA (los 18 M de parámetros que entrenamos), no el modelo base. El resultado pesa ~70 MB en disco — comparado con los 3 GB del modelo completo.

In [ ]:
trainer.save_model("./qwen-svg-lora")
tokenizer.save_pretrained("./qwen-svg-lora")

!ls -lh ./qwen-svg-lora/

## Paso K — Sanity check: generar un SVG

Antes de irnos, validamos que el modelo efectivamente aprendió algo. Le damos un prompt y vemos si genera un SVG parseable.

Nota: la primera generación puede tardar ~10 segundos porque Colab compila el grafo de inferencia la primera vez.

In [ ]:
import re

# Cargamos el modelo en modo inferencia (use_cache=True para generación rápida)
model.config.use_cache = True
model.eval()

PROMPT = "Generate svg code for an image that looks like: a red circle on a blue background"

messages = [
    {"role": "system", "content": "Respond in the following format:\n<think>\n...\n</think>\n...\n<generated_svg>\n...\n</generated_svg>"},
    {"role": "user", "content": PROMPT},
]

input_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(generated[:2000])

# Extrae el SVG de la salida (entre <svg> y </svg>)
svg_match = re.search(r"(<svg[\s\S]*?</svg>)", generated)
if svg_match:
    svg_code = svg_match.group(1)
    print("\n=== SVG extraído ===")
    print(svg_code[:500] + ("..." if len(svg_code) > 500 else ""))
else:
    print("\n⚠ No se encontró un bloque <svg>...</svg> en la salida.")

### Renderizarlo a PNG para verlo

In [ ]:
import cairosvg
from IPython.display import Image

if svg_match:
    png_bytes = cairosvg.svg2png(bytestring=svg_code.encode("utf-8"))
    display(Image(png_bytes))
else:
    print("No hay SVG para renderizar.")

## Siguiente: Paso 6 (Evaluación) y Paso 7 (Push al Hub)

Este notebook entrenó y guardó el adaptador localmente en Colab. Los próximos pasos:

- **Paso 6** — Evaluar cuantitativamente: `%` de SVGs well-formed sobre los 250 ejemplos de test, base vs fine-tuned. Tabla visual con 5 prompts curados.
- **Paso 7** — Push del adaptador al Hub en `bitneuronal/qwen-svg-coder-lora` para que otros puedan usarlo con un simple `PeftModel.from_pretrained(...)`.

Esos pasos están en notebooks separados (`eval_colab.ipynb` y `push_to_hub.ipynb`) para mantener cada uno enfocado.